In [1]:
import pandas as pd
import sqlite3 as sql

In [4]:
db = "retail_project"
conn = sql.connect("retail_project.db")

In [7]:
q = """SELECT name FROM sqlite_master WHERE type = "table" ORDER BY name; """

pd.read_sql_query(q, conn)

,name
0,categories
1,customers
2,order_items
3,orders
4,products
5,stores
6,suppliers


In [9]:
#QUERY 1
q = """SELECT order_id, customer_id,order_date,channel FROM orders 
WHERE status = 'Delivered' AND channel= 'Website' AND order_date >= '2025-11-01'AND order_date < '2025-12-01'
ORDER BY order_date DESC, order_id DESC LIMIT 10"""

pd.read_sql_query(q,conn)

,order_id,customer_id,order_date,channel
0,44176,4357,2025-11-30,Website
1,41095,8124,2025-11-30,Website
2,35572,3138,2025-11-30,Website
3,32127,1548,2025-11-30,Website
4,31788,118,2025-11-30,Website
5,29454,3581,2025-11-30,Website
6,28321,4081,2025-11-30,Website
7,24584,7044,2025-11-30,Website
8,23479,7727,2025-11-30,Website
9,22572,2596,2025-11-30,Website


In [12]:
# QUERY 2
q = """SELECT order_item_id, quantity_text,unit_price_text, 
CAST(quantity_text AS INTEGER) as quantity_num,
CAST(unit_price_text AS INTEGER) as unit_price_num,
CAST(quantity_text AS INTEGER) * CAST(unit_price_text AS FLOAT) AS gross_line_amount

FROM order_items ORDER BY order_item_id ASC LIMIT 19
"""

pd.read_sql_query(q,conn)

,order_item_id,quantity_text,unit_price_text,quantity_num,unit_price_num,gross_line_amount
0,1,1,2300.00,1,2300,2300.0
1,2,1,46100.00,1,46100,46100.0
2,3,1,9600.00,1,9600,9600.0
3,4,1,1800.00,1,1800,1800.0
4,5,1,18300.00,1,18300,18300.0
5,6,1,8800.00,1,8800,8800.0
6,7,1,169400.00,1,169400,169400.0
7,8,1,20100.00,1,20100,20100.0
8,9,1,5800.00,1,5800,5800.0
9,10,1,242300.00,1,242300,242300.0


In [16]:
#QUERY3

q = """SELECT c.customer_id, c.first_name || ' ' || c.last_name AS customer_full_name,
COUNT(o.order_id) AS order_count, 
SUM(o.net_revenue) AS total_net_revenue
FROM customers AS C INNER JOIN orders AS o ON c.customer_id = o.customer_id 
WHERE o.status = 'Delivered' GROUP BY c.customer_id, c.first_name,c.last_name ORDER BY total_net_revenue DESC, c.customer_id ASC LIMIT 5"""


pd.read_sql_query(q,conn)

,customer_id,customer_full_name,order_count,total_net_revenue
0,3024,Sara Iqbal,8,3966950.0
1,7034,Ali Shah,10,3415380.0
2,4184,Sara Siddiqui,12,3330950.0
3,4036,Hamza Ahmed,9,3317745.0
4,8476,Zain Shah,11,3276115.0


In [19]:
#QUERY4

q = """SELECT p.product_id,p.product_name,c.category_name,s.supplier_name,
SUM(oi.net_units_sold)AS total_net_units, SUM(oi.net_revenue)AS total_net_revenue,
SUM(oi.gross_profit)AS total_gross_profit

FROM order_items AS oi
INNER JOIN orders AS o ON oi.order_id = o.order_id
INNER JOIN products AS p ON oi.product_id = p.product_id
INNER JOIN categories AS c ON p.category_id = c.category_id
INNER JOIN suppliers AS s ON p.supplier_id = s.supplier_id

WHERE o.status = 'Delivered' GROUP BY p.product_id, p.product_name,c.category_name,s.supplier_name
ORDER BY total_gross_profit DESC, p.product_id ASC LIMIT 5"""

pd.read_sql_query(q,conn)

,product_id,product_name,category_name,supplier_name,total_net_units,total_net_revenue,total_gross_profit
0,193,Nova Phone 193,Smartphones,Supplier 23,504,110067940.0,36030340.0
1,89,Pulse Phone 089,Smartphones,Supplier 15,533,109986200.0,35206300.0
2,195,Aster Tablet 195,Tablets,Supplier 08,498,116125330.0,34752130.0
3,82,Vertex Laptop 082,Laptops,Supplier 18,539,124852910.0,33869710.0
4,178,Vertex Laptop 178,Laptops,Supplier 16,507,110295010.0,33839410.0


In [27]:
# QUERY5

q = """SELECT s.store_id,s.store_name, COUNT(o.order_id) AS delivered_order_count,

SUM(o.net_revenue) AS total_net_revenue, SUM(o.gross_profit) AS total_gross_profit,
AVG(o.net_revenue)AS average_order_value,
 (
 SUM(o.gross_profit) * 100.0 /NULLIF(SUM(o.net_revenue),0) 
 )AS profit_margin_pct
FROM stores AS s INNER JOIN orders AS o ON s.store_id = o.store_id
WHERE o.status = 'Delivered' GROUP BY s.store_id, s.store_name ORDER BY total_gross_profit DESC, s.store_id ASC"""

pd.read_sql_query(q,conn)


,store_id,store_name,delivered_order_count,total_net_revenue,total_gross_profit,average_order_value,profit_margin_pct
0,11,TechMart Faisalabad 11,7524,1.265792e+09,230932780.0,168233.942052,18.244131
1,1,TechMart Karachi 01,6595,1.128714e+09,208144750.0,171146.982563,18.440870
2,2,TechMart Lahore 02,5739,9.651264e+08,175999325.0,168169.790033,18.235883
3,3,TechMart Islamabad 03,4042,6.953706e+08,124518570.0,172036.261752,17.906793
4,5,TechMart Faisalabad 05,3564,6.250094e+08,114539775.0,175367.389169,18.326089
5,4,TechMart Rawalpindi 04,3004,5.029212e+08,90025230.0,167417.187084,17.900463
6,6,TechMart Multan 06,2301,3.973058e+08,72743285.0,172666.573229,18.309143
7,7,TechMart Karachi 07,2167,3.836472e+08,69707185.0,177040.694509,18.169607
8,8,TechMart Lahore 08,1750,2.998678e+08,56794045.0,171353.054286,18.939692
9,9,TechMart Islamabad 09,1736,3.010051e+08,55783200.0,173390.034562,18.532311
